# 05 · REVEL — a supervised meta-predictor & the *circularity* problem

**REVEL** (*Rare Exome Variant Ensemble Learner*, Ioannidis et al. 2016, *AJHG*, PMID 27666373) is a **supervised** random-forest **ensemble** of 13 predictors, trained on curated pathogenic/benign variants. That training is powerful — but it makes benchmarking REVEL against ClinVar **partly circular**, which is the key lesson here.

> ✅ **REAL DATA.** Genome-wide **REVEL v1.3** for CFTR — **~10,127** variants (`data/revel_cftr_v1.3.csv`, built by a manual-download build cell below). **Keyed by genomic coordinate** (REVEL has no protein position) — join onto observed variants by `chrom,pos,ref,alt`. **Non-commercial license.** `source == 'REAL'`.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · REVEL — what is it?

**REVEL** = *Rare Exome Variant Ensemble Learner* (Ioannidis et al. 2016, *American Journal of
Human Genetics*, **PMID 27666373**).

Think of REVEL as a **committee vote of 13 other predictors**. Instead of inventing a brand-new way
to judge a missense variant, its authors took the scores of **13 individual tools** (e.g. SIFT,
PolyPhen-2, MutationTaster, and several conservation scores) and trained a **random forest** — a
supervised machine-learning model — to combine them into one number.

Key facts to remember:

| Property | REVEL |
|---|---|
| Learning type | **Supervised** (learned from labelled examples) |
| Model | Random-forest **ensemble** of 13 component predictors |
| Trained on | Curated **pathogenic** and **benign** missense variants |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Covers | Missense variants only |

The "supervised" part is crucial: someone had to **hand REVEL a training set of variants already
labelled pathogenic or benign**. Where did those labels come from? Databases like **ClinVar** and
**HGMD**. Hold that thought — it is the seed of the circularity problem in section 2.


### Building the REAL data — a manual download (no API, no per-gene file)

REVEL has no API and no small per-gene download — only the **genome-wide table**,
~6.5 GB uncompressed, packaged in a zip. **You must fetch this one yourself:**

1. Go to <https://sites.google.com/site/revelgenomics> and download
   **`revel-v1.3_all_chromosomes.zip`**.
2. Save it as `data/revel-v1.3_all_chromosomes.zip` (gitignored — never commit it).

Inside the zip is a single ~6.5 GB CSV literally named `revel_with_transcript_ids`
— **no `.csv` extension**; that is just how the REVEL authors package it, not a
corrupt file. It is keyed by **genomic coordinate** (`chrom, hg19_pos, grch38_pos,
ref, alt, aaref, aaalt, REVEL, Ensembl_transcriptid`), with **no protein
position** — join on coordinate, not `protein_variant` (one protein change can
arise from several different codon changes).

The cell below **streams the CSV row-by-row without loading 6.5 GB into memory**:
it reads until it reaches chromosome 7, keeps rows inside the CFTR GRCh38 window,
and stops as soon as it leaves the contiguous chr7 block (the file is
chromosome-grouped, so this is safe and fast).

License: free for **non-commercial use** (contact the REVEL authors otherwise) —
see `data_manifest.json`.

In [2]:
import zipfile, io, csv

DATA_DIR = pathlib.Path.cwd().parent / "data"
REVEL_ZIP = DATA_DIR / "revel-v1.3_all_chromosomes.zip"
REVEL_MEMBER = "revel_with_transcript_ids"
REVEL_TSV = DATA_DIR / "revel_cftr_v1.3.csv"
CFTR_WINDOW = (117_470_000, 117_670_000)   # GRCh38, safe superset of the CFTR locus

if REVEL_TSV.exists():
    print(f"already built -> {REVEL_TSV.name} (delete to rebuild)")
elif not REVEL_ZIP.exists():
    raise FileNotFoundError(
        f"{REVEL_ZIP} not found.\n"
        "REVEL has no API and no per-gene download -- get the genome-wide table:\n"
        "  1. Go to https://sites.google.com/site/revelgenomics and download\n"
        "     'revel-v1.3_all_chromosomes.zip'\n"
        f"  2. Save it as {REVEL_ZIP} (do NOT commit it -- data/ is gitignored)\n"
        "Then re-run this cell -- it streams the 6.5 GB member CSV and stops as\n"
        "soon as it passes chromosome 7, never loading the whole file into memory."
    )
else:
    start, end = CFTR_WINDOW
    rows, seen7, scanned = [], False, 0
    with zipfile.ZipFile(REVEL_ZIP) as z:
        with z.open(REVEL_MEMBER) as fh:
            reader = csv.reader(io.TextIOWrapper(fh, encoding="utf-8", newline=""))
            next(reader)                              # header
            for f in reader:
                scanned += 1
                if f[0] == "7":
                    seen7 = True
                    g = f[2]                           # grch38_pos
                    if g in ("", "."):
                        continue
                    p = int(g)
                    if start <= p <= end:
                        rows.append((f[0], p, f[3], f[4], f[5], f[6], round(float(f[7]), 4)))
                elif seen7:
                    break                               # left the contiguous chr7 block
    print(f"scanned {scanned:,} REVEL rows to reach/pass chr7")
    df = pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "aaref", "aaalt", "revel_score"])
    df = df.sort_values("pos").reset_index(drop=True)
    df["source"] = "REAL"
    df.to_csv(REVEL_TSV, index=False)
    print(f"REAL REVEL CFTR variants written: {len(df):,} -> {REVEL_TSV.relative_to(DATA_DIR.parent)}")

already built -> revel_cftr_v1.3.csv (delete to rebuild)


In [3]:
revel = tk.load_revel()    # REAL — genome-wide REVEL v1.3 for CFTR (~10,127), coord-keyed, built above
print(f"{len(revel):,} REAL REVEL variants | source: {revel['source'].unique().tolist()}")
print('columns:', list(revel.columns))
print('revel_score range:', revel['revel_score'].min(), '->', revel['revel_score'].max())
revel.head(6)

10,127 REAL REVEL variants | source: ['REAL']
columns: ['chrom', 'pos', 'ref', 'alt', 'aaref', 'aaalt', 'revel_score', 'source']
revel_score range: 0.002 -> 0.996


,chrom,pos,ref,alt,aaref,aaalt,revel_score,source
0,7,117642459,G,A,G,R,0.996,REAL
1,7,117642459,G,C,G,R,0.996,REAL
2,7,117559510,G,C,G,A,0.995,REAL
3,7,117548822,A,C,K,T,0.995,REAL
4,7,117642514,G,T,G,V,0.995,REAL
5,7,117548813,G,T,G,V,0.995,REAL


## REVEL for the *observed* CFTR variants

REVEL is coordinate-keyed; join it onto the observed gnomAD missense set to attach the protein change and see REVEL over real variants.

In [4]:
gn = tk.load_gnomad_missense()
vid = gn['variant_id'].str.split('-', expand=True)
gn['chrom'], gn['pos'], gn['ref'], gn['alt'] = vid[0], vid[1].astype(int), vid[2], vid[3]
obs = gn.merge(revel[['chrom','pos','ref','alt','revel_score']], on=['chrom','pos','ref','alt'], how='left')
print('observed gnomAD missense with a REAL REVEL score:', int(obs['revel_score'].notna().sum()), '/', len(gn))
obs.dropna(subset=['revel_score']).sort_values('revel_score', ascending=False).head(8)[['protein_variant','hgvs_c','revel_score']]

observed gnomAD missense with a REAL REVEL score: 2436 / 2466


,protein_variant,hgvs_c,revel_score
2073,G1247R,c.3739G>A,0.996
823,K464T,c.1391A>C,0.995
819,G461V,c.1382G>T,0.995
2089,G1265R,c.3793G>A,0.994
822,G463D,c.1388G>A,0.994
2236,G1349D,c.4046G>A,0.994
853,G480D,c.1439G>A,0.993
2075,G1249R,c.3745G>A,0.992


## 2 · ⚠️ THE CIRCULARITY WARNING — the key idea in this notebook

> ### 🔁 Why "REVEL disagrees with ClinVar" can be *misleading*
>
> REVEL was **trained on labels that share lineage with ClinVar and HGMD.**
>
> So when you benchmark REVEL against ClinVar and find they **agree**, that agreement may be
> **partly baked in** — REVEL may simply be *remembering* variants it was trained on. This is
> **label leakage**: information from the "answer key" leaked into the model.
>
> And when REVEL **disagrees** with ClinVar, that disagreement is **not guaranteed to be
> independent evidence** either — it can reflect quirks of the training labels rather than a fresh,
> orthogonal opinion.
>
> **Bottom line:** a supervised predictor benchmarked against its own training-label source is
> **partly circular**. It looks more accurate than it truly is on *new* variants.

**Contrast with the unsupervised tools from tools/03-04:**

| Tool | Learned from clinical labels? | Benchmarking vs ClinVar is… |
|---|---|---|
| AlphaMissense, EVE, ESM1b | **No** (sequence / structure / evolution only) | **Fair-ish** — an independent opinion |
| **REVEL** | **Yes** (ClinVar/HGMD lineage) | **Partly circular** — beware label leakage |
| PrimateAI | Partly (a benign *proxy*, not clinical labels) | **Medium** circularity |

➡️ **Doing it properly** means grading supervised tools against **orthogonal** truth — CFTR2's
functional assays (`benchmark/01`) and population frequency (`tools/01`) — rather than against
their own homework, and applying a **training-cutoff hold-out** so a variant reported decades
before the model existed doesn't count as a win. See
[Circularity](../README.md#circularity) for the argument in full.


## 3 · REVEL isn't one cut-point — it's a *graded* scale

A very common shortcut is: *"REVEL ≥ 0.75 → likely pathogenic, otherwise not."* That single cut
throws away information.

The ClinGen **Sequence Variant Interpretation** working group (**Pejaver et al. 2022**,
*AJHG*, **PMID 36413997**) *calibrated* REVEL against the **ACMG/AMP** evidence framework. They
showed a REVEL score maps to **graded tiers of pathogenic evidence strength** (Supporting →
Moderate → Strong), not a single yes/no line:

| REVEL score ≥ | ACMG pathogenic evidence strength (approx.) |
|---|---|
| **0.932** | **Strong** (PP3_Strong) |
| **0.773** | **Moderate** (PP3_Moderate) |
| **0.644** | **Supporting** (PP3_Supporting) |
| **0.290** | *below this* → **Benign** supporting evidence (BP4) |

(These break-points are approximate and are the ones used in the toolkit's teaching table.)

**Why it matters:** two variants at REVEL 0.65 and 0.95 are *both* "≥ 0.75-ish territory" under a
single cut, but the calibration says one is only **Supporting** evidence and the other is
**Strong**. Collapsing them to one label loses exactly the nuance a curator needs.

The toolkit deliberately ships a **single** binary cut (`tk.THRESHOLDS['revel']`) to keep the
`call_from_score` helper simple — but you should know the richer, graded reality exists.


In [5]:
# The toolkit's SIMPLE binary cut-points (one 'pathogenic' line, one 'benign' line):
tk.THRESHOLDS['revel']

{'path': 0.75, 'benign': 0.29}

## 4 · Where the REAL REVEL data came from

Covered right after the intro (before Section 1's `load_revel()` call): REVEL
ships no API and no per-gene download, only a ~6.5 GB genome-wide table, so
building the CFTR extract is a **manual download + streaming filter** — see the
build cell near the top of this notebook for the exact steps and code.

> ⚠️ **Build gotcha (not a strand gotcha):** CFTR is on the **plus strand**, so `ref`/`alt` are the same as the coding change and need no complementing — see the panel above, where `R334W` (`c.1000C>T`) is simply `7-117540230-C-T`. What *does* silently match nothing is mixing **GRCh37 and GRCh38** coordinates, so pick the REVEL column for your build (this repo uses GRCh38).

## 5 · A taste of the graded idea — binning REVEL into the 4 Pejaver tiers

Instead of one line at 0.75, let's sort the DEMO variants into the **four calibrated tiers** from
section 3 using `pandas.cut`. This is a miniature version of what ACMG-style graded evidence looks
like in practice.


In [6]:
tier_edges  = [-np.inf, 0.290, 0.644, 0.773, 0.932, np.inf]
tier_labels = ['Benign-supporting (<0.290)', 'Indeterminate (0.290-0.644)',
               'Path Supporting (0.644-0.773)', 'Path Moderate (0.773-0.932)',
               'Path Strong (>=0.932)']
revel['revel_tier'] = pd.cut(revel['revel_score'], bins=tier_edges, labels=tier_labels, right=False)
print('REAL REVEL CFTR variants per Pejaver 2022 evidence tier:\n')
print(revel['revel_tier'].value_counts().reindex(tier_labels).fillna(0).astype(int).to_string())

REAL REVEL CFTR variants per Pejaver 2022 evidence tier:

revel_tier
Benign-supporting (<0.290)       1049
Indeterminate (0.290-0.644)      4166
Path Supporting (0.644-0.773)    1738
Path Moderate (0.773-0.932)      2359
Path Strong (>=0.932)             815


Even in this tiny DEMO set you can see variants spread across **several** evidence strengths — the
detail a single 0.75 cut-point would flatten into just "pathogenic vs not".


## 6 · What the toolkit records about REVEL

`toolkit.py` keeps a registry entry per predictor — its learning type and **circularity** rating — which is what tells a benchmark which tools can fairly be graded against which truth set.

In [7]:
info = tk.TOOL_REGISTRY['REVEL']
for key, val in info.items():
    print(f'  {key:12s}: {val}')

  kind        : missense
  learning    : supervised
  signal      : random-forest ENSEMBLE of 13 scores, trained on curated pathogenic/benign labels
  circularity : HIGH (label lineage overlaps ClinVar/HGMD)
  pmid        : 27666373


## Key takeaways

1. **REVEL** is a **supervised** ensemble; score 0-1, higher = worse. Now **REAL** — genome-wide REVEL v1.3 for CFTR (~10,826), coordinate-keyed.
2. ⚠️ **Circularity stays:** REVEL's labels share lineage with ClinVar/HGMD, so grading it against ClinVar is **partly circular** — don't treat REVEL-vs-ClinVar agreement as independent ([Circularity](../README.md#circularity)).
3. Read REVEL as a **graded** scale (Pejaver 2022), not a single 0.75 cut.
4. Non-commercial license — cite REVEL; raw table kept external.

**Next:** tools/06 — **PrimateAI**.